# 02 - Data Cleaning & Validation

In [ ]:
from pathlib import Path
import sys
sys.path.append('../../src')
from yolov8_tumor_detection.data_utils import list_images_and_labels, is_valid_image, validate_yolo_label

raw = Path('../../Data/Raw')
images, labels = list_images_and_labels(raw)
print(len(images), len(labels))

In [ ]:
# pairing
label_map = {l.stem: l for l in labels}
img_map = {i.stem: i for i in images}
# orphans
orphan_img = set(img_map) - set(label_map)
orphan_lab = set(label_map) - set(img_map)
print('orphan images:', len(orphan_img))
print('orphan labels:', len(orphan_lab))

In [ ]:
# validate all pairs
valid_pairs = []
bad = 0
empty_count = 0
for stem, img in img_map.items():
    if stem not in label_map:
        continue
    lab = label_map[stem]
    if not is_valid_image(img):
        bad += 1
        continue
    v = validate_yolo_label(lab, nc=4)
    if v is None:
        bad += 1
        continue
    if len(v) == 0:
        empty_count += 1
    valid_pairs.append((img, lab, v))
print('valid:', len(valid_pairs))
print('bad:', bad)
print('empty labels (no tumor):', empty_count)

In [ ]:
# save valid list
import json
out = []
for img,lab,v in valid_pairs:
    out.append({'image': str(img), 'label': str(lab), 'nboxes': len(v)})
with open('../../reports_valid.json','w') as f:
    json.dump(out,f)
print('saved')